# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. **`1 1.3333333333333333`**. В Series один пропуск (`None` → `NaN`), поэтому `isna().sum()` = 1. `fillna(0)` даёт `[1, 0, 3]`, среднее = 4/3.
2. **`2`**. После `strip()` и `lower()` получается `["a", "a", "b"]`, уникальных значений 2.
3. **`3`**. `drop_duplicates()` возвращает *новый* DataFrame, а результат никуда не записан (нет `df = ...`). Сам `df` не изменился, в нём по-прежнему 3 строки.
4. **`1005`**. `"1 000"` → `"1000"` после `replace(" ", "")`, `to_numeric` даёт 1000 и 5, сумма 1005.
5. **`[9, 10]`**. Заказ от 30 сентября попадает в месяц 9, заказ от 1 октября в месяц 10. По `dt.month` видно, что границу месяца определяет дата, а не время суток.

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [2]:
import pandas as pd

CITY_MAP = {
    "алматы": "Алматы", "almaty": "Алматы",
    "астана": "Астана", "astana": "Астана",
    "шымкент": "Шымкент", "shymkent": "Шымкент",
    "караганда": "Караганда", "актобе": "Актобе", "атырау": "Атырау",
}

def clean(path):
    df = pd.read_csv(path)
    print("прочитано строк:", len(df))

    # 1. суммы -> числа: убираем пробелы-разделители тысяч ("15 990" -> 15990)
    df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", "", regex=False))
    print("после перевода сумм в числа:", len(df))

    # 2. города: убираем пробелы, приводим к нижнему регистру, мапим через словарь
    df["city"] = df["city"].str.strip().str.lower().map(CITY_MAP)
    print("после нормализации городов:", len(df))

    # 3. полные дубли строк
    df = df.drop_duplicates()
    print("после удаления дублей:", len(df))

    # 4. даты в datetime
    df["order_date"] = pd.to_datetime(df["order_date"])
    print("после перевода дат:", len(df))

    # 5. тестовые заказы (customer_id == "TEST"). Отменённые НЕ удаляем
    df = df[df["customer_id"] != "TEST"].reset_index(drop=True)
    print("после удаления TEST:", len(df))

    # проверки
    assert df["amount"].notna().all(), "в amount есть NaN"
    assert df["city"].notna().all(), "в city есть NaN (город не попал в словарь)"
    assert df["order_id"].is_unique, "order_id не уникален"
    return df

df = clean("orders.csv")
print()
print("Итого строк:", len(df), "| городов:", df["city"].nunique())

прочитано строк: 5463
после перевода сумм в числа: 5463
после нормализации городов: 5463
после удаления дублей: 5283
после перевода дат: 5283
после удаления TEST: 5280

Итого строк: 5280 | городов: 6


**Вывод для менеджера.** После очистки остаётся **5 280 строк** из 5 463. Убрали 183: 180 полных дублей и 3 тестовых заказа с суммой 9 999 990 ₸. Все 6 городов приведены к единому написанию (`Алматы`, `астана`, `Astana`, ` Алматы` и т. п. — это один город). Отменённые заказы сохранены, по ним можно считать долю отмен.

## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [3]:
paid = df[df["status"] == "оплачен"].copy()
paid["month"] = paid["order_date"].dt.month

def change(sep, octo):
    return (octo / sep - 1) * 100

rev = paid.groupby("month")["amount"].sum()
print(f"Выручка (оплаченные) сентябрь: {rev[9]:,.0f} ₸")
print(f"Выручка (оплаченные) октябрь:  {rev[10]:,.0f} ₸")
print(f"Изменение: {change(rev[9], rev[10]):+.1f}%")
print()

rows = []
for city in sorted(paid["city"].unique()):
    p = paid[paid["city"] == city]
    s = p.loc[p["month"] == 9, "amount"].sum()
    o = p.loc[p["month"] == 10, "amount"].sum()
    rows.append((city, s, o, change(s, o)))
table = pd.DataFrame(rows, columns=["город", "сентябрь", "октябрь", "изм_%"]).sort_values("изм_%")
print(table.round({"изм_%": 1}).to_string(index=False))
worst = table.iloc[0]
print()
print(f"Сильнее всех падает {worst['город']}: {worst['изм_%']:+.1f}%")

Выручка (оплаченные) сентябрь: 35,243,410 ₸
Выручка (оплаченные) октябрь:  34,419,520 ₸
Изменение: -2.3%

    город   сентябрь    октябрь  изм_%
  Шымкент  4558650.0  4044010.0  -11.3
Караганда  3648520.0  3371680.0   -7.6
   Астана  9034220.0  8850530.0   -2.0
   Актобе  2576960.0  2540420.0   -1.4
   Алматы 14501070.0 14643020.0    1.0
   Атырау   923990.0   969860.0    5.0

Сильнее всех падает Шымкент: -11.3%


**Вывод для менеджера.** По чистым данным выручка упала всего на **2,3%**, а не на 22,7%, как казалось по сырым данным. Рост Алматы (+1,0%) и Атырау (+5,0%) почти гасит падение в остальных городах. Сильнее всех падает **Шымкент (−11,3%)**, за ним **Караганда (−7,6%)**. Именно туда нужно смотреть в первую очередь (см. п. 7: в Шымкенте с 1 октября сломалась доставка).

Сверка с таблицей «город × месяц» урока 4: таблицы урока у меня перед глазами нет (слайды в виде картинок), поэтому сверьте сами. Если там были сырые данные, цифры по Алматы и общий итог должны отличаться: в сырых данных 14 вариантов написания городов, дубли и тестовые заказы на 30 млн ₸ в Алматы. Остальные города отличаться почти не должны (кроме дублей).

## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [4]:
raw = pd.read_csv("orders.csv")

def drop_pct(d):
    d = d[d["status"] == "оплачен"]
    m = pd.to_datetime(d["order_date"]).dt.month
    r = d.groupby(m)["amount"].sum()
    return (r[10] / r[9] - 1) * 100

step0 = raw.copy()
step0["amount"] = pd.to_numeric(step0["amount"].str.replace(" ", "", regex=False))
step1 = step0.drop_duplicates()
step2 = step1[step1["customer_id"] != "TEST"]

journal = [
    ("Суммы в числа (убрали пробелы)", 0, drop_pct(step0)),
    ("Удалили полные дубли", len(step0) - len(step1), drop_pct(step1)),
    ("Удалили тестовые заказы TEST", len(step1) - len(step2), drop_pct(step2)),
]
print("| Шаг очистки | Убрано строк | «Падение» выручки сентябрь→октябрь |")
print("|---|---|---|")
prev = None
for name, n, p in journal:
    print(f"| {name} | {n} | {p:+.1f}% |")

| Шаг очистки | Убрано строк | «Падение» выручки сентябрь→октябрь |
|---|---|---|
| Суммы в числа (убрали пробелы) | 0 | -22.7% |
| Удалили полные дубли | 180 | -19.6% |
| Удалили тестовые заказы TEST | 3 | -2.3% |


**Вывод для менеджера.** Сильнее всего цифру исказили **3 тестовых заказа**: их удаление сдвинуло «падение» с −19,6% до **−2,3%**, потому что каждый на 9 999 990 ₸ (всего ~30 млн ₸, это вдвое больше месячной выручки всего Алматы). Дубли дали ещё ~3 п. п., а города и даты на общую выручку не влияют.

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [5]:
na = df["delivery_days"].isna()
print("Пропусков в delivery_days:", na.sum(), f"({na.mean()*100:.1f}% строк)")
print()
print("По городам (всего пропусков | доля пропусков в городе):")
by_city = pd.DataFrame({"пропусков": df[na]["city"].value_counts(),
                        "доля_%": (df.groupby("city")["delivery_days"].apply(lambda s: s.isna().mean()) * 100).round(1)})
print(by_city.sort_values("пропусков", ascending=False).to_string())
print()
print("По статусам (пропусков | доля):")
by_st = pd.DataFrame({"пропусков": df[na]["status"].value_counts(),
                      "доля_%": (df.groupby("status")["delivery_days"].apply(lambda s: s.isna().mean()) * 100).round(1)})
print(by_st.to_string())
print()

sh = df[df["city"] == "Шымкент"]["delivery_days"]
a = (sh.dropna() <= 2).mean() * 100
b = (sh.fillna(sh.median()) <= 2).mean() * 100
c = (sh.fillna(0) <= 2).mean() * 100
print("Доля доставки не дольше 2 дней в Шымкенте:")
print(f"  (а) dropna:          {a:.1f}%")
print(f"  (б) fillna(медиана={sh.median():.0f}): {b:.1f}%")
print(f"  (в) fillna(0):       {c:.1f}%")

Пропусков в delivery_days: 158 (3.0% строк)

По городам (всего пропусков | доля пропусков в городе):
           пропусков  доля_%
city                        
Алматы            63     3.0
Астана            40     3.0
Караганда         24     4.6
Шымкент           22     2.8
Атырау             5     3.5
Актобе             4     1.0

По статусам (пропусков | доля):
         пропусков  доля_%
status                    
оплачен        144     3.0
отменён         14     2.6

Доля доставки не дольше 2 дней в Шымкенте:
  (а) dropna:          29.8%
  (б) fillna(медиана=5): 29.0%
  (в) fillna(0):       31.8%


**Вывод для менеджера.** Пропусков 158 (3,0% строк). Больше всего их в абсолютных числах в Алматы (63) и Астане (40), потому что там больше заказов. В долях разница небольшая (1–5% по городам, чуть выше в Караганде, 4,6%). По статусам доля одинакова (оплачен 3,0%, отменён 2,6%), то есть пропуски не привязаны к отменам.

Доля заказов, доставленных за ≤2 дня, в Шымкенте: (а) 29,8%, (б) 29,0%, (в) 31,8%. **Честный вариант (а), `dropna`**: мы не знаем срок этих заказов и не выдумываем его. `fillna(0)` записывает «доставили мгновенно» и завышает долю быстрых доставок (31,8%). `fillna(медиана)` берёт медиану 5 дней, которая сама искажена сломавшейся с октября доставкой (в сентябре медиана была бы 2), поэтому занижает долю. Разница небольшая (3% пропусков), но принцип важен.

## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [6]:
# Исправленный код
df_ai = pd.read_csv("orders.csv")
df_ai = df_ai.drop_duplicates()                      # ошибка 1: результат не был присвоен
df_ai["city"] = df_ai["city"].str.strip().str.lower().map(CITY_MAP)   # ошибка 2: нет strip и Latin-вариантов не сведены словарём
print(df_ai["city"].nunique(), len(df_ai))

# Для сравнения: что давал исходный код AI
bad = pd.read_csv("orders.csv")
bad.drop_duplicates()
bad["city"] = bad["city"].str.lower()
print("код AI:", bad["city"].nunique(), len(bad))
print("варианты городов у AI:", sorted(bad["city"].unique()))

6 5283
код AI: 10 5463
варианты городов у AI: [' алматы', 'almaty', 'astana ', 'shymkent', 'актобе', 'алматы', 'астана', 'атырау', 'караганда', 'шымкент']


**Две ошибки в коде AI:**
1. `df.drop_duplicates()` не записан обратно (`df = df.drop_duplicates()`), поэтому дубли остались, и строк по-прежнему 5 463.
2. `str.lower()` недостаточно: остались пробелы (`" алматы"`, `"astana "`) и латинские `astana`, `almaty`, `shymkent`. Нужно `str.strip().str.lower()` и словарь соответствий (`map`), который сводит латиницу к русскому написанию. Тогда получается 6 городов вместо 10.

После исправления: 6 городов, 5 283 строки (ещё 3 тестовых заказа убирает `clean()` из п. 1, итого 5 280).

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [7]:
wd = df["order_date"].dt.day_name().value_counts()
print(wd.to_string())
print()
lo, hi = wd.idxmin(), wd.idxmax()
print(f"Меньше всего заказов: {lo} ({wd.min()}), больше всего: {hi} ({wd.max()})")
print(f"{lo} меньше, чем {hi}, на {(1 - wd.min()/wd.max())*100:.1f}%")

order_date
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684

Меньше всего заказов: Monday (684), больше всего: Thursday (786)
Monday меньше, чем Thursday, на 13.0%


**Вывод для менеджера.** Заказов меньше всего в **понедельник (684)**, это на **13,0% меньше**, чем в самый загруженный день, четверг (786). Воскресенье тоже слабое (697). Остальные дни очень ровные (773–786). Для графика курьеров: в понедельник и воскресенье можно держать на ~10–13% меньше смен, в остальные дни смены одинаковые. Резкого пика в выходные нет, перераспределять курьеров на субботу не нужно.

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [8]:
sh = df[(df["city"] == "Шымкент") & df["delivery_days"].notna()].copy()
sh["month"] = sh["order_date"].dt.month
print("Средний срок доставки в Шымкенте:")
print(f"  сентябрь: {sh.loc[sh['month']==9,  'delivery_days'].mean():.2f} дня")
print(f"  октябрь:  {sh.loc[sh['month']==10, 'delivery_days'].mean():.2f} дня")

# Первый заказ со сроком >= 5 дней (фильтр + sort_values по дате)
slow = sh[sh["delivery_days"] >= 5].sort_values("order_date")
first = slow.iloc[0]["order_date"]
print("\nПервый «медленный» заказ (>=5 дней):", first)

# Проверяем, что дальше медленнее 5 бывает ТОЛЬКО так: быстрых после этой даты нет
fast_after = sh[(sh["order_date"] >= first) & (sh["delivery_days"] < 5)]
fast_before = sh[sh["order_date"] < first].sort_values("order_date")
print("Быстрых (<5 дней) заказов после этой даты:", len(fast_after))
print("Последний быстрый заказ до неё:", fast_before.iloc[-1]["order_date"],
      "| макс. срок до даты:", fast_before["delivery_days"].max())
print("Минимальный срок после даты:", sh[sh["order_date"] >= first]["delivery_days"].min())

# Другие города
oth = df[(df["city"] != "Шымкент") & df["delivery_days"].notna()].copy()
oth["после"] = oth["order_date"] >= first
print("\nДругие города: срок доставки до/после этой даты")
print(oth.groupby("после")["delivery_days"].agg(["mean", "min", "max", "count"]).round(2).to_string())

Средний срок доставки в Шымкенте:


  сентябрь: 2.03 дня
  октябрь:  6.54 дня

Первый «медленный» заказ (>=5 дней): 2026-10-01 10:00:00
Быстрых (<5 дней) заказов после этой даты: 0
Последний быстрый заказ до неё: 2026-09-30 22:00:00 | макс. срок до даты: 3.0
Минимальный срок после даты: 5.0

Другие города: срок доставки до/после этой даты
       mean  min  max  count
после                       
False  2.00  1.0  3.0   2175
True   2.01  1.0  3.0   2193


**Сообщение менеджеру.** Средний срок доставки в Шымкенте вырос с **2,03 дня в сентябре до 6,54 дня в октябре**. Это не плавный рост, а скачок: **с 1 октября 2026** в Шымкенте нет ни одного заказа быстрее 5 дней (до этого срок был 1–3 дня). В остальных городах после этой даты ничего не изменилось (≈2,0 дня и до, и после, диапазон 1–3). Значит, проблема локальная, в самом Шымкенте.

Что проверить в первую очередь: что поменялось в Шымкенте на рубеже сентября и октября: смена службы доставки или склада/хаба, потеря курьеров, изменение маршрута или закрытие пункта выдачи. Это же объясняет падение выручки Шымкента на 11,3% (п. 2): медленная доставка → отмены и отток клиентов.

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [9]:
import numpy as np

def to_amount(s):
    """Строки -> числа. Понимает '15 990', '15990.0', '1 200,50'; остальное -> NaN."""
    s = (s.astype(str)
           .str.replace(r"\s", "", regex=True)      # любые пробелы, в т.ч. неразрывный
           .str.replace(",", ".", regex=False))      # запятая как десятичный разделитель
    return pd.to_numeric(s, errors="coerce")         # непонятное -> NaN

test = pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])
print(to_amount(test).tolist())

raw = pd.read_csv("orders.csv")
mine = to_amount(raw["amount"]).sum()
lesson = pd.to_numeric(raw["amount"].str.replace(" ", "", regex=False)).sum()
print(f"Сумма to_amount: {mine:,.0f} | сумма по уроку: {lesson:,.0f} | совпало: {np.isclose(mine, lesson)}")
print("NaN в столбце amount после to_amount:", to_amount(raw["amount"]).isna().sum())

[15990.0, 15990.0, 1200.5, nan]
Сумма to_amount: 109,883,360 | сумма по уроку: 109,883,360 | совпало: True
NaN в столбце amount после to_amount: 0


**Вывод.** Функция даёт `[15990.0, 15990.0, 1200.5, nan]`: все три формата распознаны, а «нет данных» превратилось в NaN, а не в ошибку. На столбце `amount` из orders.csv сумма совпадает с суммой из урока, NaN нет. `errors="coerce"` нужен, чтобы одна плохая строка не ломала всю загрузку, но после него обязательно проверять `isna().sum()`: иначе мусор тихо выпадет из суммы.